<a href="https://colab.research.google.com/github/summer-2022/ai-agent-systems/blob/main/movie-agents/notebooks/movie_memory_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install Library

In [ ]:
!pip install openai

### Load API Key

In [ ]:
from google.colab import userdata
from openai import OpenAI

# Colab Secret 불러오기
api_key = userdata.get("OPENAI_API_KEY")

# OpenAI client 생성
client = OpenAI(api_key=api_key)

### Initialize Chat Memory

In [ ]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a movie recommendation chatbot.\n"
            "Rules:\n"
            "- Recommend a movie only when the user asks for recommendation.\n"
            "- Only recommend one movie at a time.\n"
            "- If the user says they like a movie, respond positively without recommending a new one.\n"
            "- Keep answers short."
        )
    }
]

memory = {
    "favorite_genres": [],
    "watched_movies": []
}

### Movie Database

In [ ]:
movie_db = {
    "SF": ["Blade Runner 2049", "The Matrix", "Dune", "The Martian"],
    "Romance": ["About Time", "The Notebook", "La La Land"],
    "Action": ["John Wick", "Top Gun Maverick"]
}

### Helper Functions

In [ ]:
def update_memory(user_text):
    if "SF" in user_text:
        memory["favorite_genres"].append("SF")

    if "Inception" in user_text:
        memory["watched_movies"].append("Inception")

    if "Interstellar" in user_text:
        memory["watched_movies"].append("Interstellar")

### Call LLM

In [ ]:
def call_ai():

    context = f"""
Favorite genres: {memory['favorite_genres']}
Watched movies: {memory['watched_movies']}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages + [{"role": "system", "content": context}]
    )

    message = response.choices[0].message.content
    messages.append({"role": "assistant", "content": message})

    return message

### Chat Function

In [ ]:
def chat(user_text):

    messages.append({"role": "user", "content": user_text})

    update_memory(user_text)

    reply = call_ai()

    print("User:", user_text)
    print("AI:", reply)

### Chat Test

In [ ]:
while True:
    user_text = input("Send a message to the LLM... ")

    if user_text.lower() in ["q", "quit", "exit"]:
        print("챗봇을 종료합니다.")
        break

    chat(user_text)

Send a message to the LLM... Romance 
User: Romance 
AI: Could you please tell me what you're in the mood for?
Send a message to the LLM... Romance movie
User: Romance movie
AI: I recommend "The Notebook." It's a classic romantic drama you might enjoy.
Send a message to the LLM... I have watched it
User: I have watched it
AI: That's great! Let me know if you'd like another recommendation.
Send a message to the LLM... Can you recommend me another romance movie 
User: Can you recommend me another romance movie 
AI: Sure! How about "Pride and Prejudice"? It's a beautiful adaptation of the classic novel.
Send a message to the LLM... Okay Thank you
User: Okay Thank you
AI: You're welcome! If you have any more questions or need another recommendation, just let me know!
Send a message to the LLM... q
챗봇을 종료합니다.


## Korean version

In [ ]:
# -------------------------------
# 영화 추천 챗봇 전체 코드
# -------------------------------

# 대화 기록 저장
messages = [
    {
        "role": "system",
        "content": (
            "너는 사용자의 영화 취향을 기억해서 개인화된 추천을 해주는 영화 추천 챗봇이야. "
            "항상 한국어로 자연스럽고 친절하게 답변해. "
            "이미 본 영화는 절대 다시 추천하지 마."
        )
    }
]

# 사용자 메모리
memory = {
    "favorite_genres": [],
    "watched_movies": []
}

# 간단한 영화 추천 DB
movie_db = {
    "SF": [
        "블레이드 러너 2049",
        "매트릭스",
        "듄",
        "마션",
        "그래비티",
        "엑스 마키나"
    ],
    "로맨스": [
        "어바웃 타임",
        "노트북",
        "라라랜드",
        "비포 선라이즈"
    ],
    "액션": [
        "존 윅",
        "탑건: 매버릭",
        "매드 맥스: 분노의 도로",
        "미션 임파서블"
    ],
    "스릴러": [
        "셔터 아일랜드",
        "세븐",
        "겟 아웃",
        "나이브스 아웃"
    ]
}

# 영화 제목 인식용 후보
known_movies = [
    "인셉션",
    "인터스텔라",
    "블레이드 러너 2049",
    "매트릭스",
    "듄",
    "마션",
    "그래비티",
    "엑스 마키나",
    "어바웃 타임",
    "노트북",
    "라라랜드",
    "존 윅",
    "탑건: 매버릭",
    "셔터 아일랜드",
    "세븐",
    "겟 아웃",
    "나이브스 아웃"
]

# 장르 인식용 키워드
genre_map = {
    "SF": ["sf", "SF", "공상과학"],
    "로맨스": ["로맨스"],
    "액션": ["액션"],
    "스릴러": ["스릴러"],
    "애니메이션": ["애니", "애니메이션"],
    "공포": ["공포", "호러"]
}


def extract_genres(text):
    found = []
    lower_text = text.lower()
    for genre, keywords in genre_map.items():
        for keyword in keywords:
            if keyword.lower() in lower_text:
                found.append(genre)
                break
    return found


def extract_movies(text):
    found = []
    for movie in known_movies:
        if movie in text:
            found.append(movie)
    return found


def update_memory(user_text):
    # 좋아하는 장르 추출
    genres = extract_genres(user_text)
    for genre in genres:
        if genre not in memory["favorite_genres"]:
            memory["favorite_genres"].append(genre)

    # 이미 본 영화 추출
    watched_signals = ["봤어", "이미 봤어", "봤다", "본 영화", "시청했어"]
    if any(signal in user_text for signal in watched_signals):
        movies = extract_movies(user_text)
        for movie in movies:
            if movie not in memory["watched_movies"]:
                memory["watched_movies"].append(movie)


def build_candidate_list():
    candidates = []
    for genre in memory["favorite_genres"]:
        if genre in movie_db:
            for movie in movie_db[genre]:
                if movie not in memory["watched_movies"] and movie not in candidates:
                    candidates.append(movie)
    return candidates


def build_memory_context():
    genres = ", ".join(memory["favorite_genres"]) if memory["favorite_genres"] else "없음"
    watched = ", ".join(memory["watched_movies"]) if memory["watched_movies"] else "없음"
    candidates = build_candidate_list()
    candidate_text = ", ".join(candidates) if candidates else "없음"

    return f"""
현재까지 기억한 사용자 정보:
- 좋아하는 장르: {genres}
- 이미 본 영화: {watched}
- 추천 가능 후보: {candidate_text}

응답 규칙:
1. 이미 본 영화는 절대 추천하지 마.
2. 사용자가 추천을 요청하면 추천 가능 후보를 바탕으로 추천해.
3. 사용자가 자신이 좋아하는 장르나 이미 본 영화를 물으면 위 정보를 정확히 말해.
4. 한국어로 짧고 자연스럽게 답해.
"""


def call_ai():
    memory_context = build_memory_context()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages + [
            {
                "role": "system",
                "content": memory_context
            }
        ]
    )

    ai_message = response.choices[0].message.content
    messages.append({"role": "assistant", "content": ai_message})
    return ai_message


def chat(user_text):
    # 사용자 메시지 저장
    messages.append({"role": "user", "content": user_text})

    # 메모리 업데이트
    update_memory(user_text)

    # AI 응답 생성
    ai_reply = call_ai()

    print(f"User: {user_text}")
    print(f"AI: {ai_reply}")
    print("-" * 60)

In [ ]:
chat("나는 SF 영화를 좋아해")

User: 나는 SF 영화를 좋아해
AI: 좋아요! 당신은 SF 영화를 좋아하시네요. 현재까지 본 영화는 없고, 추천할 수 있는 영화 목록이 있어요. 어떤 영화가 궁금하신가요? 예를 들어, 블레이드 러너 2049, 매트릭스, 듄, 마션, 그래비티, 엑스 마키나 중에서 추천해 드릴 수 있어요!
------------------------------------------------------------


In [ ]:
chat("인셉션이랑 인터스텔라는 이미 봤어")

User: 인셉션이랑 인터스텔라는 이미 봤어
AI: 그럼 다른 SF 영화를 추천해드릴게요! 블레이드 러너 2049나 그래비티는 어떠세요? 흥미로운 스토리와 비주얼로 매력을 느낄 수 있을 거예요! 선택하시면 더 많은 정보를 제공해 드릴 수 있습니다.
------------------------------------------------------------


In [ ]:
chat("이번주말에 뭐 볼지 추천해 줄래?")


User: 이번주말에 뭐 볼지 추천해 줄래?
AI: 이번 주말에 볼 영화로는 **블레이드 러너 2049**를 추천해 드릴게요! 시각적으로 멋진 장면과 깊이 있는 이야기가 정말 인상적이에요. 활용해 보시면 좋을 것 같아요!
------------------------------------------------------------


In [ ]:
chat("내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?")

User: 내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?
AI: 당신이 좋아하는 장르는 SF이고, 이미 본 영화는 **인셉션**과 **인터스텔라**예요. 지금 추천 가능한 영화로는 **블레이드 러너 2049**, **매트릭스**, **듄**, **마션**, **그래비티**, **엑스 마키나**가 있어요! 어떤 영화가 궁금하신가요?
------------------------------------------------------------
